# 2일차 파트 1 예제: LLM API 기초

교재의 예제 1-1부터 1-12까지를 실행하는 노트북입니다. 셀 첫 줄의 `# 예제 1-N`이 교재의 예제 번호입니다.
위에서부터 차례로 실행합니다. 모델의 응답은 실행할 때마다 조금씩 다릅니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다.

어떤 API를 쓸지는 환경 변수로 정합니다.

| 쓰는 API | 정할 환경 변수 |
| --- | --- |
| Anthropic | `ANTHROPIC_API_KEY` |
| 사내 API(OpenAI 호환) | `LLM_PROVIDER=openai`, `OPENAI_BASE_URL`, `OPENAI_API_KEY`, `LLM_MODEL_SMALL`, `LLM_MODEL_DEFAULT`, `LLM_MODEL_LARGE` |

환경 변수는 Jupyter를 시작하기 전에 정해 두는 것이 가장 안전합니다. 아직 정하지 않았다면 아래 첫 셀이 값을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북에도 저장되지 않습니다.

In [1]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

사용하는 API: anthropic


In [2]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data/2일차'
SYSTEM = open(f'{DATA}/분류_규칙.txt', encoding='utf-8').read()

# 100만 토큰당 달러 (입력, 출력). Anthropic 공개 가격, 2026-11-03 확인.
# 사내 API 를 쓴다면 사내 과금 기준으로 바꿉니다. 과금이 없으면 (0, 0) 으로 둡니다.
PRICE = {'소형': (0.10, 0.50), '기본': (2.00, 10.00), '대형': (4.00, 20.00)}


def cost_usd(input_tokens, output_tokens, model='기본'):
    """토큰 수와 단가로 요금(달러)을 계산합니다."""
    input_price, output_price = PRICE[model]
    return (input_tokens * input_price + output_tokens * output_price) / 1_000_000


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

소형 모델: claude-haiku-5-5
기본 모델: claude-sonnet-5-5
대형 모델: claude-opus-5-5


준비 완료


## 예제 1-1 첫 호출과 응답 읽기

요청에 꼭 필요한 값은 모델 이름, 최대 출력 토큰, 메시지 목록입니다. 응답에서는 응답 텍스트, 토큰 수, 종료 이유를 읽습니다.
먼저 SDK를 직접 불러 요청과 응답의 실제 모양을 봅니다. 두 셀 중 지금 쓰는 API에 해당하는 셀만 실행됩니다.

In [3]:
# 예제 1-1 (Anthropic SDK 로 직접 호출)
question = '3호기 로드락 도어 인터록 알람이 세 번 발생했습니다. 이 문의를 한 문장으로 요약해 주세요.'

if llm.provider() == 'anthropic':
    import anthropic

    client = anthropic.Anthropic()            # 환경 변수 ANTHROPIC_API_KEY 를 읽습니다
    response = client.messages.create(
        model=llm.model_name('기본'),
        max_tokens=1024,                      # 최대 출력 토큰
        messages=[{'role': 'user', 'content': question}],
    )
    print('블록 종류:', [block.type for block in response.content])
    print('응답:', ''.join(block.text for block in response.content if block.type == 'text'))
    print('입력 토큰:', response.usage.input_tokens)
    print('출력 토큰:', response.usage.output_tokens)
    print('종료 이유:', response.stop_reason)
    print('답한 모델:', response.model)
else:
    print('Anthropic 을 쓰지 않으므로 건너뜁니다.')

블록 종류: ['thinking', 'text']
응답: 3호기 로드락 도어 인터록 알람이 세 번 발생하여, 이에 대한 원인 확인 및 조치를 요청하는 문의입니다.

원문에 발생 시점, 알람 코드, 공정 상태 같은 세부 정보가 없어 요약에는 담지 못했습니다. 이 정보를 알려주시면 더 구체적으로 정리해 드리겠습니다.
입력 토큰: 60
출력 토큰: 242
종료 이유: end_turn
답한 모델: claude-sonnet-5-5


In [4]:
# 예제 1-1 (OpenAI 호환 SDK 로 직접 호출)
if llm.provider() == 'openai':
    from openai import OpenAI

    client = OpenAI(base_url=os.environ['OPENAI_BASE_URL'], api_key=os.environ['OPENAI_API_KEY'])
    response = client.chat.completions.create(
        model=llm.model_name('기본'),
        max_tokens=1024,                      # 최대 출력 토큰
        messages=[{'role': 'user', 'content': question}],
    )
    choice = response.choices[0]
    print('응답:', choice.message.content)
    print('입력 토큰:', response.usage.prompt_tokens)
    print('출력 토큰:', response.usage.completion_tokens)
    print('종료 이유:', choice.finish_reason)
    print('답한 모델:', response.model)
else:
    print('OpenAI 호환 API 를 쓰지 않으므로 건너뜁니다.')

OpenAI 호환 API 를 쓰지 않으므로 건너뜁니다.


두 SDK는 값의 이름과 위치가 다를 뿐 담긴 내용은 같습니다. 이후 예제는 이 차이를 감춘 `llm.send()`를 씁니다.
`llm.send()`는 메시지 목록을 한 번 보내고 응답, 토큰 수, 종료 이유, 모델 이름, 걸린 시간을 `Reply`로 돌려줍니다.

In [5]:
# 예제 1-1 (llm.send 로 같은 호출)
reply = llm.send([{'role': 'user', 'content': question}])
print('응답:', reply.text)
print('입력 토큰:', reply.input_tokens, '· 출력 토큰:', reply.output_tokens)
print('종료 이유:', reply.stop_reason, '· 모델:', reply.model, f'· {reply.seconds:.1f}초')

응답: 3호기 로드락 도어에서 인터록 알람이 세 차례 발생했다는 문의입니다.

원인 분석이나 점검 항목이 필요하시면 알람 코드, 발생 시각, 직전 작업 내용을 알려주세요.
입력 토큰: 60 · 출력 토큰: 295
종료 이유: end · 모델: claude-sonnet-5-5 · 2.8초


## 예제 1-2 최대 출력 토큰을 낮춰 잘린 응답 만들기

최대 출력 토큰을 일부러 작게 주고 종료 이유가 어떻게 바뀌는지 봅니다.

In [6]:
# 예제 1-2
reply = llm.send([{'role': 'user', 'content': question}], max_tokens=15)
print('종료 이유:', reply.stop_reason)
print('출력 토큰:', reply.output_tokens)
print('응답:', repr(reply.text))

종료 이유: max_tokens
출력 토큰: 15
응답: ''


## 예제 1-3 같은 질문을 두 번 보내기

같은 모델에 같은 질문을 두 번 보내 응답을 비교합니다.

In [7]:
# 예제 1-3
for number in [1, 2]:
    reply = llm.send([{'role': 'user', 'content': question}])
    print(number, reply.text)

1 3호기 로드락 도어에서 인터록 알람이 세 차례 발생하여 이에 대한 확인 및 조치를 요청하는 문의입니다.

원인 파악까지 요청하시려면 알람 발생 시각, 알람 코드, 당시 공정 상태를 함께 알려 주시면 좋습니다.


2 3호기 로드락 도어에서 인터록 알람이 세 차례 발생하여, 원인 확인과 조치가 필요하다는 문의입니다.


## 예제 1-4 시스템 프롬프트로 분류 규칙 고정하기

분류 규칙은 `data/2일차/분류_규칙.txt`에 있습니다. 이 글을 시스템 프롬프트로 주고, 메시지에는 문의만 넣습니다.

In [8]:
# 예제 1-4
print(SYSTEM)
print('-' * 40)

inquiry = 'id: Q900\n문의: 3호기 로드락 도어 인터록 알람이 세 번 연속 발생했습니다'
reply = llm.send([{'role': 'user', 'content': inquiry}], system=SYSTEM)
print(reply.text)
print('입력 토큰:', reply.input_tokens)

너는 반도체 공장 현장에서 들어오는 문의를 분류하는 도우미다.

문의를 아래 다섯 가지 중 하나로 분류한다.
- 문서안내: 문서, 자료, 양식의 위치나 최신본을 묻는 문의
- 절차안내: 어떻게 하는지(신청, 승인, 작업 순서, 담당, 기준)를 묻는 문의
- 데이터요청: 측정값, 집계, 로그 같은 데이터를 달라는 문의
- 긴급: 지금 일어나고 있는 설비 이상, 알람, 정지를 알리는 문의
- 기타: 위 넷에 해당하지 않는 글

두 가지에 걸치면 긴급, 데이터요청, 절차안내, 문서안내 순으로 앞선 것을 고른다.
urgent 는 즉시 또는 오늘 안에 대응해 달라는 문의면 true, 아니면 false 다. 긴급은 항상 true 다.

출력은 아래 형식의 JSON 한 줄만 쓴다. 설명이나 코드 블록 표시를 붙이지 않는다.
{"id": "문의 id", "category": "분류", "urgent": true 또는 false}

----------------------------------------


{"id": "Q900", "category": "긴급", "urgent": true}
입력 토큰: 482


In [9]:
# 예제 1-4 (규칙은 그대로 두고 문의만 바꿉니다)
inquiry_2 = 'id: Q901\n문의: PM 체크리스트 최신본 위치를 알고 싶습니다'
print(llm.send([{'role': 'user', 'content': inquiry_2}], system=SYSTEM).text)

{"id": "Q901", "category": "문서안내", "urgent": false}


## 예제 1-5 서버는 앞 대화를 기억하지 않는다

먼저 사실 하나를 알려 주고, 다음 호출에서 그 사실을 물어봅니다. 앞 대화를 함께 보내지 않은 경우와 보낸 경우를 비교합니다.

In [10]:
# 예제 1-5
first = {'role': 'user', 'content': '우리 팀 설비 점검 요일은 목요일입니다. 기억해 주세요.'}
first_reply = llm.send([first], max_tokens=300)
print('첫 응답:', first_reply.text)

follow_up = {'role': 'user', 'content': '우리 팀 설비 점검 요일이 무슨 요일이라고 했죠? 요일만 답해 주세요.'}

# (가) 새 질문만 보냅니다
alone = llm.send([follow_up], max_tokens=300)
print('\n(가) 새 질문만 보냄 →', alone.text)

# (나) 앞 질문과 응답을 함께 보냅니다
history = [first, {'role': 'assistant', 'content': first_reply.text}, follow_up]
together = llm.send(history, max_tokens=300)
print('(나) 앞 대화를 함께 보냄 →', together.text)
print('\n입력 토큰: (가)', alone.input_tokens, '· (나)', together.input_tokens)

첫 응답: 네, 팀 설비 점검 요일이 **목요일**이라는 점을 기억해 두겠습니다.

이 대화 안에서는 계속 참고할 수 있습니다. 다만 새 대화에서도 기억될지는 사용 중인 환경의 메모리 기능 지원 여부에 따라 달라서, 중요한 일정이라면 별도로 메모나 캘린더에 등록해 두시길 권합니다.

점검 체크리스트나 안내 공지문, 알림 문구가 필요하시면 말씀해 주세요.



(가) 새 질문만 보냄 → 죄송하지만 저는 팀의 설비 점검 요일을 알지 못합니다. 이 대화에서는 그런 내용을 들은 적이 없고, 이전 대화도 볼 수 없습니다. 점검 일정이 적힌 공지나 메시지를 알려 주시면 확인해 드리겠습니다.


(나) 앞 대화를 함께 보냄 → 목요일입니다.

입력 토큰: (가) 50 · (나) 278


## 예제 1-6 대화가 길어지면 입력 토큰이 늘어난다

대화를 네 턴 이어 가며 호출마다 입력 토큰을 봅니다. 그다음 최근 한 턴만 남겨 보내 봅니다.

In [11]:
# 예제 1-6
questions = ['반도체 공정에서 PM 이 무엇의 줄임말인지 한 문장으로 알려 주세요.',
             '그것을 정해진 주기로 하는 이유를 한 문장으로 알려 주세요.',
             '주기를 놓치면 생길 수 있는 문제를 한 가지만 알려 주세요.',
             '지금까지 답한 내용을 한 문장으로 정리해 주세요.']

history = []
for turn, text in enumerate(questions, 1):
    history.append({'role': 'user', 'content': text})
    reply = llm.send(history, model='소형', max_tokens=500)       # 이력 전체를 보냅니다
    history.append({'role': 'assistant', 'content': reply.text})  # 응답도 이력에 넣습니다
    print(f'{turn}번째 호출 · 입력 토큰 {reply.input_tokens}')
print('\n마지막 응답:', reply.text)

1번째 호출 · 입력 토큰 48


2번째 호출 · 입력 토큰 183


3번째 호출 · 입력 토큰 309


4번째 호출 · 입력 토큰 415

마지막 응답: 반도체 공정의 PM(예방 정비)은 설비 고장과 오염 누적을 막기 위해 정해진 주기마다 점검·교체·청소를 수행하는 활동이며, 이 주기를 놓치면 파티클과 부품 마모가 쌓여 웨이퍼 불량과 수율 저하가 생길 수 있습니다.


In [12]:
# 예제 1-6 (최근 한 턴과 마지막 질문만 보냅니다)
# history 는 [질문1, 답1, 질문2, 답2, 질문3, 답3, 질문4, 답4] 입니다.
recent = history[-4:-1]          # [질문3, 답3, 질문4]
reply = llm.send(recent, model='소형', max_tokens=500)
print('입력 토큰:', reply.input_tokens)
print('응답:', reply.text)

입력 토큰: 148
응답: 주기를 놓치면 설비에 파티클과 마모가 쌓여 웨이퍼 불량과 수율 저하가 생길 수 있습니다.


## 예제 1-7 요금 계산과 사전 추정

응답에 담긴 토큰 수로 실제 요금을 계산합니다. 그다음 문의 전체를 보내기 전에 요금을 미리 추정합니다.

In [13]:
# 예제 1-7
reply = llm.send([{'role': 'user', 'content': inquiry}], system=SYSTEM, model='소형')
print(f'입력 {reply.input_tokens} · 출력 {reply.output_tokens} 토큰')
print(f"요금 ${cost_usd(reply.input_tokens, reply.output_tokens, '소형'):.6f}")

입력 482 · 출력 26 토큰
요금 $0.000061


In [14]:
# 예제 1-7 (보내기 전에 입력 토큰을 셉니다)
counted = llm.count_tokens([{'role': 'user', 'content': inquiry}], system=SYSTEM, model='소형')
print('보내기 전에 센 값:', counted)
print('실제로 보낸 뒤의 값:', reply.input_tokens)
# OpenAI 호환 API 는 보내기 전에 세는 표준 방법이 없어 None 이 나옵니다.

보내기 전에 센 값: 482
실제로 보낸 뒤의 값: 482


In [15]:
# 예제 1-7 (문의 전체의 요금을 추정합니다)
inquiries = pd.read_csv(f'{DATA}/문의_예제.csv', dtype=str, keep_default_na=False)
valid = inquiries[inquiries['문의내용'].str.strip() != '']
print('전체', len(inquiries), '건 · 빈 문의', len(inquiries) - len(valid), '건')


def prompt_of(row):
    return f"id: {row['id']}\n문의: {row['문의내용']}"


def count_one(row):
    return llm.count_tokens([{'role': 'user', 'content': prompt_of(row)}], system=SYSTEM, model='소형')


rows = [row for _, row in valid.iterrows()]
if llm.provider() == 'anthropic':
    with ThreadPoolExecutor(max_workers=4) as pool:
        input_counts = list(pool.map(count_one, rows))
    input_total = sum(input_counts)
    print(f'입력 합계 {input_total} · 한 건 평균 {input_total / len(rows):.0f} · 최대 {max(input_counts)}')
else:
    # 세어 주는 기능이 없으면 몇 건을 실제로 보내 평균을 내고 건수를 곱합니다
    sample = [llm.send([{'role': 'user', 'content': prompt_of(row)}], system=SYSTEM, model='소형')
              for row in rows[:5]]
    average = sum(reply.input_tokens for reply in sample) / len(sample)
    input_total = round(average * len(rows))
    print(f'표본 5건 평균 {average:.0f} 토큰 → 입력 합계 추정 {input_total}')

EXPECTED_OUTPUT = 40     # 응답은 JSON 한 줄이므로 한 건에 40토큰으로 가정합니다
for alias in ['소형', '기본', '대형']:
    estimate = cost_usd(input_total, EXPECTED_OUTPUT * len(rows), alias)
    print(f'{alias} 모델로 {len(rows)}건: 약 ${estimate:.4f}')

전체 100 건 · 빈 문의 2 건


입력 합계 47648 · 한 건 평균 486 · 최대 771
소형 모델로 98건: 약 $0.0067
기본 모델로 98건: 약 $0.1345
대형 모델로 98건: 약 $0.2690


## 예제 1-8 정답이 있는 표본으로 모델 등급 비교하기

정답 라벨이 있는 문의 20건을 등급별로 분류해 정확도, 시간, 요금을 비교합니다.

In [16]:
# 예제 1-8
answers = pd.read_csv(f'{DATA}/문의_예제_정답.csv', dtype=str).set_index('id')
# 엣지 케이스 12건과 보통 사례 8건을 표본으로 씁니다. 쉬운 문의만 고르면 모델 차이가 드러나지 않습니다.
hard_ids = answers.index[answers['난이도'] == '엣지']
easy_ids = answers.index[answers['난이도'] == '보통'][:8]
sample = valid[valid['id'].isin(hard_ids.union(easy_ids))]
print('표본', len(sample), '건 · 엣지 케이스', len(hard_ids), '건')


def classify(row, model):
    """문의 한 건을 분류해 (분류, Reply) 를 돌려줍니다. JSON 으로 읽지 못하면 분류는 None 입니다."""
    reply = llm.send([{'role': 'user', 'content': prompt_of(row)}], system=SYSTEM, model=model)
    try:
        return json.loads(reply.text)['category'], reply
    except (json.JSONDecodeError, KeyError, TypeError):
        return None, reply


MODELS = ['소형', '기본', '대형']      # 대형 모델을 쓰지 않으려면 목록에서 뺍니다
# 여러 등급에 같은 모델 이름을 넣었다면 한 번만 비교합니다
MODELS = [alias for index, alias in enumerate(MODELS)
          if llm.model_name(alias) not in [llm.model_name(earlier) for earlier in MODELS[:index]]]
comparison = {}
for model in MODELS:
    results = [classify(row, model) for _, row in sample.iterrows()]
    correct = sum(category == answers.loc[row_id, 'category']
                  for (category, _), row_id in zip(results, sample['id']))
    seconds = sum(reply.seconds for _, reply in results)
    input_tokens = sum(reply.input_tokens or 0 for _, reply in results)
    output_tokens = sum(reply.output_tokens or 0 for _, reply in results)
    comparison[model] = results
    print(f'{model}: 정답 {correct}/{len(sample)} · {seconds:.1f}초 · 입력 {input_tokens} · 출력 {output_tokens}'
          f' · ${cost_usd(input_tokens, output_tokens, model):.4f}')

표본 20 건 · 엣지 케이스 12 건


소형: 정답 18/20 · 22.0초 · 입력 9867 · 출력 1794 · $0.0019


기본: 정답 20/20 · 35.9초 · 입력 9867 · 출력 2667 · $0.0464


대형: 정답 18/20 · 48.9초 · 입력 9867 · 출력 2251 · $0.0845


In [17]:
# 예제 1-8 (틀린 건을 직접 읽어 봅니다)
for model in MODELS:
    for (category, _), (_, row) in zip(comparison[model], sample.iterrows()):
        expected = answers.loc[row['id'], 'category']
        if category != expected:
            print(f"[{model}] {row['id']} 정답 {expected} · 모델 {category} · {row['문의내용'][:50]}")

[소형] Q026 정답 문서안내 · 모델 긴급 · 8호기가 조금 이상해서 우선 매뉴얼부터 보고 싶습니다
[소형] Q033 정답 절차안내 · 모델 긴급 · E-201 알람이 났을 때 임시 조치 방법을 알려 주세요
[대형] Q026 정답 문서안내 · 모델 긴급 · 8호기가 조금 이상해서 우선 매뉴얼부터 보고 싶습니다
[대형] Q060 정답 문서안내 · 모델 데이터요청 · 지난달 스크랩 사유 코드 정리본이 있나요


## 예제 1-9 오류를 일부러 일으켜 종류 확인하기

`llm.send()`는 실패하면 `CallError`를 냅니다. `status`는 상태 코드, `kind`는 오류 종류, `retryable`은 재시도해 볼 만한 오류인지를 알려 줍니다.

In [18]:
# 예제 1-9
hello = [{'role': 'user', 'content': '안녕하세요'}]
cases = {
    '없는 모델 이름': dict(messages=hello, model='no-such-model-0000'),
    '틀린 키': dict(messages=hello, api_key='wrong-key'),
    '빈 메시지 목록': dict(messages=[]),
    '타임아웃 0.01초': dict(messages=hello, timeout=0.01),
}
for name, options in cases.items():
    try:
        llm.send(max_tokens=50, **options)
        print(f'{name}: 오류 없이 끝남')
    except llm.CallError as error:
        print(f'{name}: 상태 코드 {error.status} · 종류 {error.kind} · 재시도할 만한가 {error.retryable}')

없는 모델 이름: 상태 코드 404 · 종류 not_found · 재시도할 만한가 False


틀린 키: 상태 코드 401 · 종류 auth · 재시도할 만한가 False


빈 메시지 목록: 상태 코드 400 · 종류 bad_request · 재시도할 만한가 False
타임아웃 0.01초: 상태 코드 None · 종류 timeout · 재시도할 만한가 True


## 예제 1-10 재시도와 백오프를 넣은 `ask()` 만들기

재시도할 만한 오류면 기다렸다가 재시도해고, 그렇지 않으면 바로 넘기는 함수를 만듭니다.
`send`와 `sleep`을 인자로 받게 해 두면, 실제 API 대신 mock 호출을 넣어 동작을 확인할 수 있습니다.

In [19]:
# 예제 1-10
def ask(prompt, system=None, max_tokens=300, tries=3, send=llm.send, sleep=time.sleep):
    """질문 하나를 보냅니다. 재시도할 만한 오류면 1초, 2초로 기다리며 tries 번까지 보냅니다."""
    messages = [{'role': 'user', 'content': prompt}]
    for attempt in range(tries):
        try:
            return send(messages, system=system, max_tokens=max_tokens)
        except llm.CallError as error:
            if not error.retryable or attempt == tries - 1:
                raise                      # 재시도해도 소용없는 오류, 또는 마지막 시도
            wait_seconds = 2 ** attempt    # 1초, 2초, 4초 …
            print(f'  {error.kind} → {wait_seconds}초 뒤 재시도합니다')
            sleep(wait_seconds)

In [20]:
# 예제 1-10 (mock 호출로 동작을 확인합니다. API 를 부르지 않습니다)
import checks

fake = checks.FakeSend(checks.error('rate_limit', 429), checks.error('server', 529), checks.good_reply())
waits = []
reply = ask('아무 질문', send=fake, sleep=waits.append)     # 실제로 기다리지 않고 기다릴 시간만 기록합니다
print('보낸 횟수:', fake.calls, '· 대기하려던 시간:', waits, '· 결과:', reply.stop_reason)

fake = checks.FakeSend(checks.error('not_found', 404))
try:
    ask('아무 질문', send=fake, sleep=waits.append)
except llm.CallError as error:
    print('404 는 보낸 횟수:', fake.calls, '· 바로 넘긴 오류:', error.kind)

  rate_limit → 1초 뒤 재시도합니다
  server → 2초 뒤 재시도합니다
보낸 횟수: 3 · 대기하려던 시간: [1, 2] · 결과: end
404 는 보낸 횟수: 1 · 바로 넘긴 오류: not_found


In [21]:
# 예제 1-10 (실제 API 로 한 번 보냅니다)
print(ask(inquiry, system=SYSTEM).text)

{"id": "Q900", "category": "긴급", "urgent": true}


## 예제 1-11 오류 없이 왔지만 쓸 수 없는 응답 필터링하기

잘린 응답, 거절된 응답, JSON으로 읽을 수 없는 응답은 예외를 내지 않습니다. 응답을 받은 뒤 코드로 검사합니다.

In [22]:
# 예제 1-11
def check_reply(reply):
    """쓸 수 없는 응답이면 사유를, 쓸 수 있으면 None 을 돌려줍니다."""
    if reply.stop_reason == 'max_tokens':
        return '잘림'
    if reply.stop_reason == 'refusal':
        return '거절'
    try:
        parsed = json.loads(reply.text)
    except json.JSONDecodeError:
        return '파싱 오류'
    if not isinstance(parsed, dict) or 'category' not in parsed:
        return '파싱 오류'
    return None


normal = ask(inquiry, system=SYSTEM)
print('보통 응답 →', check_reply(normal), '·', repr(normal.text))

cut = ask(inquiry, system=SYSTEM, max_tokens=8)     # 일부러 잘리게 합니다
print('max_tokens=8 →', check_reply(cut), '·', repr(cut.text))

보통 응답 → None · '{"id": "Q900", "category": "긴급", "urgent": true}'


max_tokens=8 → 잘림 · '{"id": "Q900'


## 예제 1-12 배치 처리: 입력 검사, 병렬 처리, 실패 목록

보내기 전에 입력을 필터링하고, 여러 건을 동시에 보내고, 성공과 실패를 따로 모읍니다.

In [23]:
# 예제 1-12
def is_valid(text):
    """보낼 만한 문의인지 돌려줍니다. 여기서는 빈 문의만 필터링합니다."""
    return isinstance(text, str) and len(text.strip()) >= 2


def process_one(row):
    """문의 한 건을 처리해 ('성공', 결과) 또는 ('실패', (id, 사유)) 와 로그 한 줄을 돌려줍니다."""
    try:
        reply = ask(prompt_of(row), system=SYSTEM)
    except llm.CallError as error:
        return '실패', (row['id'], error.kind), None
    log = (row['id'], reply.model, reply.input_tokens, reply.output_tokens, reply.stop_reason)
    reason = check_reply(reply)
    if reason:
        return '실패', (row['id'], reason), log
    return '성공', json.loads(reply.text), log


batch = inquiries.head(30)
to_send = [row for _, row in batch.iterrows() if is_valid(row['문의내용'])]
print('대상', len(batch), '건 · 보내기 전에 필터링한 건', len(batch) - len(to_send), '건')

started = time.time()
with ThreadPoolExecutor(max_workers=4) as pool:        # 네 건씩 동시에 보냅니다
    outcomes = list(pool.map(process_one, to_send))
elapsed = time.time() - started

success = [result for status, result, _ in outcomes if status == '성공']
failed = [result for status, result, _ in outcomes if status == '실패']
logs = [log for _, _, log in outcomes if log]
print(f'성공 {len(success)}건 · 실패 {len(failed)}건 {failed} · {elapsed:.1f}초')
print('결과 첫 줄:', success[0])
print('로그 첫 줄:', logs[0])

대상 30 건 · 보내기 전에 필터링한 건 1 건


성공 28건 · 실패 1건 [('Q026', '잘림')] · 11.1초
결과 첫 줄: {'id': 'Q001', 'category': '문서안내', 'urgent': False}
로그 첫 줄: ('Q001', 'claude-sonnet-5-5', 477, 27, 'end')


In [24]:
# 예제 1-12 (실패한 건만 다시 처리합니다)
# 잘린 건은 최대 출력 토큰을 올려 재시도합니다. 다른 사유는 사람이 확인할 목록으로 남깁니다.
by_id = {row['id']: row for row in to_send}
still_failed = []
for row_id, reason in failed:
    if reason != '잘림':
        still_failed.append((row_id, reason))
        continue
    reply = ask(prompt_of(by_id[row_id]), system=SYSTEM, max_tokens=1024)
    logs.append((row_id, reply.model, reply.input_tokens, reply.output_tokens, reply.stop_reason))
    second_reason = check_reply(reply)
    if second_reason:
        still_failed.append((row_id, second_reason))
    else:
        success.append(json.loads(reply.text))
        print(row_id, '재시도해 성공 · 출력 토큰', reply.output_tokens)
print('성공', len(success), '건 · 남은 실패', still_failed)

Q026 재시도해 성공 · 출력 토큰 404
성공 29 건 · 남은 실패 []


In [25]:
# 예제 1-12 (정답과 비교하고 실제 요금을 계산합니다)
correct = sum(result['category'] == answers.loc[result['id'], 'category'] for result in success)
input_tokens = sum(log[2] or 0 for log in logs)
output_tokens = sum(log[3] or 0 for log in logs)
print(f'정답 {correct}/{len(success)}')
print(f"입력 {input_tokens} · 출력 {output_tokens} · 실제 요금 ${cost_usd(input_tokens, output_tokens, '기본'):.4f}")

정답 28/29
입력 14886 · 출력 1988 · 실제 요금 $0.0497


## 보충 A. Claude의 thinking과 `effort`

Anthropic API를 쓸 때만 실행되는 셀입니다. 응답에 thinking 블록이 어떻게 들어오는지, `effort` 값에 따라 출력 토큰이 어떻게 달라지는지 봅니다.

In [26]:
# 예제 보충A
if llm.provider() == 'anthropic':
    import anthropic

    client = anthropic.Anthropic()
    puzzle = ('한 로트는 웨이퍼 25장입니다. 장비 한 대는 한 번에 6장까지 처리하고 한 번에 40분이 걸립니다. '
              '장비 두 대로 3로트를 처리하는데, 서로 다른 로트의 웨이퍼를 한 번에 섞어 넣을 수 없습니다. '
              '모두 처리하는 데 걸리는 최소 시간은 몇 분입니까? 답과 근거를 두 문장으로 알려 주세요.')
    for effort in ['low', 'high']:
        started = time.time()
        response = client.messages.create(
            model=llm.model_name('기본'),
            max_tokens=4000,
            output_config={'effort': effort},
            messages=[{'role': 'user', 'content': puzzle}],
        )
        answer = ''.join(block.text for block in response.content if block.type == 'text')
        print(f"effort={effort}: {time.time() - started:.1f}초 · 출력 {response.usage.output_tokens} 토큰"
              f" · 블록 {[block.type for block in response.content]} · 응답 {answer!r}")
else:
    print('Anthropic 을 쓰지 않으므로 건너뜁니다.')

effort=low: 3.3초 · 출력 261 토큰 · 블록 ['text'] · 응답 '각 로트는 25장이므로 한 장비에서 6장씩 처리하면 최소 5번(6+6+6+6+1)이 필요하고, 3로트면 총 15번의 배치가 필요한데, 서로 다른 로트를 섞을 수 없으므로 이 15번은 줄일 수 없습니다. 장비 두 대가 병렬로 처리하면 15번을 한 대 8번, 다른 대 7번으로 나눌 수 있어 최소 8번×40분 = **320분**입니다.\n\n(배치 분배 예: 장비 A는 로트1 5번 + 로트2 3번, 장비 B는 로트2 2번 + 로트3 5번으로 하면 8번과 7번이 되어 가능합니다.)'


effort=high: 4.1초 · 출력 428 토큰 · 블록 ['thinking', 'text'] · 응답 '**답은 320분입니다.** 한 로트(25장)는 한 번에 6장까지만 처리되고 로트끼리 섞을 수 없어서 최소 ⌈25/6⌉=5번이 필요하므로, 3로트는 총 15번(배치)이고, 장비 2대로 나누면 한 대가 최소 ⌈15/2⌉=8번을 맡아 8×40분=320분이 걸립니다. 한 로트를 두 장비에 나눠 돌려도 되므로 이 하한은 실제로 달성할 수 있습니다.'
